In [ ]:
import numpy as np
import pandas as pd
import json, sys
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from scipy.special import expit

sys.path.append("/Proyecto/Value-disagreement/Python/Utilities")
import Dict_Object #, text_cleansing

In [ ]:
# Loadiong Data
value_set = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all.csv", sep='|')

value_train_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_train.csv", delimiter=',', dtype=int)
value_val_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_val.csv", delimiter=',', dtype=int)
value_test_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_test.csv", delimiter=',', dtype=int)

train_df = value_set.iloc[value_train_set]
val_df = value_set.iloc[value_val_set]
test_df = value_set.iloc[value_test_set]
test_df

In [ ]:
def predict_on_val(model_ckpt, tokenizer_dir, val_hf_tokenized):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_ckpt,
        num_labels=1
    )
    model.resize_token_embeddings(len(tokenizer))

    args = TrainingArguments(
        output_dir=".",
        do_train=False,
        do_eval=False,
        do_predict=True,
        per_device_eval_batch_size=32,
        seed=0
    )

    trainer = Trainer(
        model=model,
        args=args,
        tokenizer=tokenizer
    )

    out = trainer.predict(val_hf_tokenized)

    logits = out.predictions.squeeze()
    probs = expit(logits)

    df = pd.DataFrame({
        "id": val_hf_tokenized["id"],
        "value": val_hf_tokenized["value"],
        "label": np.array(out.label_ids).astype(int),
        "prob": probs.astype(float)
    })

    return df

In [ ]:
# Load models
roberta_model = AutoModelForSequenceClassification.from_pretrained("/Proyecto/Value-disagreement/Python/Models/Results/roberta-base_table-valueALL_seed0/checkpoint-11617/")
deberta_model = AutoModelForSequenceClassification.from_pretrained("/Proyecto/Value-disagreement/Python/Models/Results/microsoft/deberta-v3-base_table-valueALL_seed4/checkpoint-8712/")

#roberta_model.eval()
#deberta_model.eval()

#roberta_tokenizer = AutoTokenizer.from_pretrained("/Proyecto/Value-disagreement/Python/Models/Results/roberta-base_table-valueALL_seed0/checkpoint-11617/")
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
# Add special value tokens (10 values)
roberta_tokenizer.add_special_tokens({"additional_special_tokens": [f"<{x}>" for x in Dict_Object.ValueConstants.SCHWARTZ_VALUES]})

#deberta_tokenizer = AutoTokenizer.from_pretrained("/Proyecto/Value-disagreement/Python/Models/Results/microsoft/deberta-v3-base_table-valueALL_seed4/checkpoint-8712/")
deberta_tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
# Add special value tokens (10 values)
deberta_tokenizer.add_special_tokens({"additional_special_tokens": [f"<{x}>" for x in Dict_Object.ValueConstants.SCHWARTZ_VALUES]})

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move models to GPU
roberta_model.to(device)
deberta_model.to(device)

In [ ]:
# Thresholds per value
with open("/Proyecto/Value-disagreement/Python/Models/best_thresholds_per_model.json") as f:
    thresholds = json.load(f)

value_labels = list(thresholds.keys())  # value order
n_classes = len(value_labels)

# Best model per value
with open("/Proyecto/Value-disagreement/Python/Models/best_model_per_value.json") as f:
    best_model_per_value = json.load(f)

In [ ]:
best_model_per_value

In [ ]:
"""for k in best_model_per_value.keys():
    best_model_per_value[k]='ensemble'
best_model_per_value['POWER']='microsoft_deberta-v3-base'
best_model_per_value"""

In [ ]:
# Save
"""with open("best_model_per_value.json", "w") as f:
    json.dump(best_model_per_value, f, indent=2)"""

## ON TEST SET

In [ ]:
test_df

In [ ]:
# Formated input strings for each row: <VALUE> [SEP] TEXT
roberta_formatted_texts = [f"<{row['value']}> {roberta_tokenizer.sep_token} {row['scenario']}"
                           for _, row in test_df.iterrows()]

deberta_formatted_texts = [f"<{row['value']}> {deberta_tokenizer.sep_token} {row['scenario']}"
                           for _, row in test_df.iterrows()]

In [ ]:
batch_size = 128
preds, probs, roberta_prb, deberta_prb, ensemble_prb = [],[],[],[],[]

for i in range(0, len(deberta_formatted_texts), batch_size):
    roberta_batch_texts = roberta_formatted_texts[i:i+batch_size]
    deberta_batch_texts = deberta_formatted_texts[i:i+batch_size]
    batch_values = test_df["value"].iloc[i:i+batch_size].tolist()
    batch_labels = test_df["label"].iloc[i:i+batch_size].tolist()
    
    # Tokenize
    roberta_inputs = roberta_tokenizer(roberta_batch_texts, padding='max_length', max_length=256, truncation=True, return_tensors="pt").to(device)
    deberta_inputs = deberta_tokenizer(deberta_batch_texts, padding='max_length', max_length=256, truncation=True, return_tensors="pt").to(device)
    
    # Add batch-specific labels
    #roberta_inputs["labels"] = torch.tensor(batch_labels, dtype=torch.float).to(device)
    #deberta_inputs["labels"] = torch.tensor(batch_labels, dtype=torch.float).to(device)

    with torch.no_grad():
        roberta_logits = roberta_model(**roberta_inputs).logits
        deberta_logits = deberta_model(**deberta_inputs).logits
    
    # Convert logits to probabilities
    roberta_probs = torch.sigmoid(roberta_logits).squeeze(-1)
    deberta_probs = torch.sigmoid(deberta_logits).squeeze(-1)
    ensemble_probs = (roberta_probs + deberta_probs) / 2

    #roberta_prb.append(roberta_probs)
    #deberta_prb.append(deberta_probs)
    #ensemble_prb.append(ensemble_probs)
    
    for j, value in enumerate(batch_values):
        best_model = best_model_per_value[value]
        #best_model = 'roberta-base'
        #best_model = 'microsoft_deberta-v3-base'
        #best_model = 'ensemble'
        threshold = thresholds[best_model][value]

        if best_model == "roberta-base":
            prob = roberta_probs[j].item()
        elif best_model == "microsoft_deberta-v3-base":
            prob = deberta_probs[j].item()
        else:
            prob = ensemble_probs[j].item()
            
        pred = int(prob >= threshold)

        #print(f"Value: {value}, Threshold Used: {threshold}, Prob: {prob:.3f}, Pred: {pred}")
        
        preds.append(pred)
        probs.append(prob)

        # for generating the ensemble tresholds
        #roberta_prb.append(roberta_probs[j])
        #deberta_prb.append(deberta_probs[j])
        #ensemble_prb.append(ensemble_probs[j])
        
# Save predictions
test_df["pred"] = preds
test_df["prob"] = probs
test_df

In [ ]:
# for generating the ensemble tresholds

#test_df["roberta_probs"] = torch.tensor(roberta_prb).cpu().numpy()
#test_df["deberta_probs"] = torch.tensor(deberta_prb).cpu().numpy()
#test_df["ensemble_probs"] = torch.tensor(ensemble_prb).cpu().numpy()

#test_df.to_csv(r"/Proyecto/Value-disagreement/Python/Models/ensemble.probs.csv",
#               header=True, index=False, quotechar='"', sep="|", escapechar="|"
#               )

In [ ]:
import matplotlib.pyplot as plt

# Assuming roberta_probs and deberta_probs are from the full test set
plt.hist(roberta_probs.cpu().numpy().flatten(), bins=50, alpha=0.6, label="RoBERTa")
plt.hist(deberta_probs.cpu().numpy().flatten(), bins=50, alpha=0.6, label="DeBERTa")
plt.title("Model Output Calibration (Sigmoid Probabilities)")
plt.xlabel("Probability")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
print("RoBERTa:")
print(f"Mean prob: {roberta_probs.mean():.3f}, Std: {roberta_probs.std():.3f}")

print("DeBERTa:")
print(f"Mean prob: {deberta_probs.mean():.3f}, Std: {deberta_probs.std():.3f}")

print("Ensemble:")
print(f"Mean prob: {ensemble_probs.mean():.3f}, Std: {ensemble_probs.std():.3f}")

In [ ]:
test_df['pred'].value_counts()

In [ ]:
test_df.groupby("value")["pred"].value_counts()

In [ ]:
for val in test_df["value"].unique():
    val_df = test_df[test_df["value"] == val]
    print(f"{val}: Positives = {val_df['pred'].sum()}, Total = {len(val_df)}")

In [ ]:
report = classification_report(
    test_df["label"], 
    test_df["pred"], 
    target_names=["Negative", "Positive"]
)
print("=== Classifclassification_reportication Report ===")
print(report)

In [ ]:
print("\n=== PER-VALUE F1 ===")
for val in test_df["value"].unique():
    val_df = test_df[test_df["value"] == val]
    f1 = f1_score(val_df["label"], val_df["pred"], zero_division=0)
    print(f"{val:>20}: F1 = {f1:.3f}")

## ON NEW DATA

In [ ]:
df_all = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/cleaned/deba_comments_final_sample.csv",
                      sep="|"
                      )
df_all['value']='ACHIEVEMENT'
df_all

In [ ]:
# Formated input strings for each row: <VALUE> [SEP] TEXT
roberta_formatted_texts = [f"<{row['value']}> {roberta_tokenizer.sep_token} {row['body_cleand']}"
                           for _, row in df_all.iterrows()]

deberta_formatted_texts = [f"<{row['value']}> {deberta_tokenizer.sep_token} {row['body_cleand']}"
                           for _, row in df_all.iterrows()]

In [ ]:
batch_size = 128
preds, probs, roberta_prb, deberta_prb, ensemble_prb = [],[],[],[],[]

for i in range(0, len(deberta_formatted_texts), batch_size):
    roberta_batch_texts = roberta_formatted_texts[i:i+batch_size]
    deberta_batch_texts = deberta_formatted_texts[i:i+batch_size]
    batch_values = df_all["value"].iloc[i:i+batch_size].tolist()
    #batch_labels = test_df["label"].iloc[i:i+batch_size].tolist()
    
    # Tokenize
    roberta_inputs = roberta_tokenizer(roberta_batch_texts, padding='max_length', max_length=256, truncation=True, return_tensors="pt").to(device)
    deberta_inputs = deberta_tokenizer(deberta_batch_texts, padding='max_length', max_length=256, truncation=True, return_tensors="pt").to(device)
    
    # Add batch-specific labels
    #roberta_inputs["labels"] = torch.tensor(batch_labels, dtype=torch.float).to(device)
    #deberta_inputs["labels"] = torch.tensor(batch_labels, dtype=torch.float).to(device)

    with torch.no_grad():
        roberta_logits = roberta_model(**roberta_inputs).logits
        deberta_logits = deberta_model(**deberta_inputs).logits
    
    # Convert logits to probabilities
    roberta_probs = torch.sigmoid(roberta_logits).squeeze(-1)
    deberta_probs = torch.sigmoid(deberta_logits).squeeze(-1)
    ensemble_probs = (roberta_probs + deberta_probs) / 2

    #roberta_prb.append(roberta_probs)
    #deberta_prb.append(deberta_probs)
    #ensemble_prb.append(ensemble_probs)
    
    for j, value in enumerate(batch_values):
        best_model = best_model_per_value[value]
        #best_model = 'roberta-base'
        #best_model = 'microsoft_deberta-v3-base'
        #best_model = 'ensemble'
        threshold = thresholds[best_model][value]

        if best_model == "roberta-base":
            prob = roberta_probs[j].item()
        elif best_model == "microsoft_deberta-v3-base":
            prob = deberta_probs[j].item()
        else:
            prob = ensemble_probs[j].item()
            
        pred = int(prob >= threshold)

        #print(f"Value: {value}, Threshold Used: {threshold}, Prob: {prob:.3f}, Pred: {pred}")
        
        preds.append(pred)
        probs.append(prob)

        # for generating the ensemble tresholds
        #roberta_prb.append(roberta_probs[j])
        #deberta_prb.append(deberta_probs[j])
        #ensemble_prb.append(ensemble_probs[j])
        
# Save predictions
df_all["pred"] = preds
df_all["prob"] = probs
df_all

In [ ]:
print("RoBERTa:")
print(f"Mean prob: {roberta_probs.mean():.3f}, Std: {roberta_probs.std():.3f}")

print("DeBERTa:")
print(f"Mean prob: {deberta_probs.mean():.3f}, Std: {deberta_probs.std():.3f}")

print("Ensemble:")
print(f"Mean prob: {ensemble_probs.mean():.3f}, Std: {ensemble_probs.std():.3f}")

In [ ]:
df_all['pred'].value_counts()

In [ ]:
df_all[df_all['pred']==1].sample(15)[['body_cleand','pred','prob']]

In [ ]:
['id', 'author','topic', 'body_cleand','value', 'pred', 'prob']

In [ ]:
df_all.columns

In [ ]:
# === Input text(s) ===
test_df = [
    "I yelled at my friend in public",
    "I gave money to someone in need",
    ...
]

# === Tokenize ===
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

# === Predict (no gradients needed) ===
with torch.no_grad():
    roberta_logits = roberta_model(**inputs).logits
    deberta_logits = deberta_model(**inputs).logits

# === Convert logits to probabilities ===
roberta_probs = torch.sigmoid(roberta_logits)
deberta_probs = torch.sigmoid(deberta_logits)

# === Ensemble probabilities ===
ensemble_probs = (roberta_probs + deberta_probs) / 2

# === Apply thresholds per value ===
threshold_tensor = torch.tensor([thresholds[v] for v in value_labels])
binary_preds = (ensemble_probs >= threshold_tensor).int()

# === Format predictions ===
for i, text in enumerate(texts):
    active_values = [value_labels[j] for j in range(n_classes) if binary_preds[i][j] == 1]
    print(f"\nText: {text}\nPredicted Values: {active_values}")


In [ ]:
del roberta_model, deberta_model, roberta_inputs, deberta_inputs
torch.cuda.empty_cache()
import gc
gc.collect()